# DeepQuality Colab 展示笔记本

本笔记本用于在 Google Colab 中完整展示 DeepQuality 项目：数据读取、配置说明、S-DDAE 基线训练、SS-DDFAE 半监督训练、后处理搜索、checkpoint 评估、在线推理模拟和结果可视化。

主线实验使用 Debutanizer 数据集，默认使用优化后的 `window_size=30`、`quality_delay=8`、`label_ratio=1.0` 展示完整训练效果。笔记本会先执行诊断训练，只用于估算耗时，不展示低轮次诊断指标。

## 1. Colab 运行前准备

本笔记本直接从 GitHub 克隆项目：

```text
https://github.com/HT3301601278/DeepQuality.git
```

运行下面单元格后，项目会被克隆到 `/content/DeepQuality`。

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/HT3301601278/DeepQuality.git'
PROJECT_DIR = Path('/content/DeepQuality')

if PROJECT_DIR.exists():
    print('项目目录已存在，拉取最新代码')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    print('克隆 DeepQuality 仓库')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print('当前项目目录:', Path.cwd())

## 2. 安装依赖

安装项目本身和依赖。Colab 通常已预装 PyTorch，下面命令会按项目配置补齐依赖。

In [ ]:
%pip install -q -e .

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. 项目结构检查

确认 Colab 中已经包含代码、配置、数据和说明文件。

In [ ]:
!find . -maxdepth 3 -type f | sort | sed 's#^./##' | head -n 80

## 4. 数据集展示

默认数据集是 Debutanizer。输入变量为 `u1` 到 `u7`，目标质量变量为 `y`。

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/Debutanizer_Data.csv')
print('shape:', df.shape)
display(df.head())
display(df.describe().T)

## 5. 配置文件展示

`configs/base.yaml` 负责数据路径、划分比例、窗口长度、质量延迟和输出目录。模型配置文件覆盖基础配置中的同名字段。

In [ ]:
from pathlib import Path
import yaml

for config_path in ['configs/base.yaml', 'configs/sddae_single_scale.yaml', 'configs/ss_ddfae.yaml']:
    print('\n' + '=' * 80)
    print(config_path)
    print('=' * 80)
    print(Path(config_path).read_text(encoding='utf-8'))

## 6. 诊断运行与参数推荐

本节不按设备名称预设训练轮数，而是先执行最小诊断训练，再根据实测耗时推荐正式训练参数。

推荐流程如下：

1. 读取配置文件中的完整训练轮数。
2. 执行一次最小 S-DDAE 诊断训练，估算 S-DDAE 单轮耗时。
3. 使用诊断 S-DDAE checkpoint 执行一次最小 SS-DDFAE 诊断训练，估算 SS-DDFAE 单轮耗时。
4. 根据目标训练总时长计算缩放系数。
5. 输出本次正式训练推荐参数。

默认 `TARGET_TOTAL_MINUTES = 40`，表示希望两类模型的正式训练总耗时控制在约 40 分钟内。若希望获得更充分的训练效果，可调大该值；若只做课堂或答辩演示，可调小该值。

当前主线参数来自本项目实验复核：`window_size=30`、`quality_delay=8` 在验证集选择下得到更好的测试表现。

诊断训练只用于测速，诊断 checkpoint、指标表和图片会在测速结束后删除，避免混入正式展示结果。

In [ ]:
import os
import platform
import subprocess
import shutil
import time
from pathlib import Path

import pandas as pd
import torch
import yaml

TARGET_TOTAL_MINUTES = 40
LABEL_RATIO = 1.0
WINDOW_SIZE = 30
QUALITY_DELAY = 8
LATENT_DIM = 32
DIAG_BASELINE_NAME = 'diagnose_runtime_sddae_probe'
DIAG_IMPROVED_NAME = 'diagnose_runtime_ss_ddfae_probe'

Path('/content/.matplotlib').mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = '/content/.matplotlib'

def load_yaml(path):
    with open(path, 'r', encoding='utf-8') as file:
        return yaml.safe_load(file)

def detect_runtime():
    runtime = {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'training_device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'device_name': 'CPU',
        'cuda_available': torch.cuda.is_available(),
        'gpu_memory_gb': 0.0,
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        runtime['device_name'] = props.name
        runtime['gpu_memory_gb'] = round(props.total_memory / 1024**3, 2)
    return runtime

def run_timed(command, label):
    print(f'开始诊断: {label}')
    started = time.perf_counter()
    completed = subprocess.run(command, check=True, text=True, capture_output=True)
    elapsed = time.perf_counter() - started
    print(f'{label} 耗时: {elapsed:.2f} 秒')
    return elapsed

def remove_diagnostic_artifacts(*names):
    for name in names:
        for path in Path('artifacts').glob(f'**/{name}*'):
            if path.is_file():
                path.unlink()

sddae_config = load_yaml('configs/sddae_single_scale.yaml')
ss_config = load_yaml('configs/ss_ddfae.yaml')
default_pretrain_epochs = int(sddae_config['training']['pretrain_epochs'])
default_finetune_epochs = int(sddae_config['training']['finetune_epochs'])
default_ss_epochs = int(ss_config['training']['epochs'])

runtime = detect_runtime()

sddae_probe_seconds = run_timed([
    'python', '-m', 'deep_quality.cli.train_sddae',
    '--config', 'configs/sddae_single_scale.yaml',
    '--label-ratio', str(LABEL_RATIO),
    '--window-size', str(WINDOW_SIZE),
    '--quality-delay', str(QUALITY_DELAY),
    '--latent-dim', str(LATENT_DIM),
    '--pretrain-epochs', '1',
    '--finetune-epochs', '1',
    '--output-name', DIAG_BASELINE_NAME,
], 'S-DDAE 1轮预训练 + 1轮微调')

ss_probe_seconds = run_timed([
    'python', '-m', 'deep_quality.cli.train_ss_ddfae',
    '--config', 'configs/ss_ddfae.yaml',
    '--baseline-checkpoint', f'artifacts/checkpoints/{DIAG_BASELINE_NAME}.pt',
    '--label-ratio', str(LABEL_RATIO),
    '--window-size', str(WINDOW_SIZE),
    '--quality-delay', str(QUALITY_DELAY),
    '--latent-dim', str(LATENT_DIM),
    '--epochs', '1',
    '--output-name', DIAG_IMPROVED_NAME,
], 'SS-DDFAE 1轮训练')

remove_diagnostic_artifacts(DIAG_BASELINE_NAME, DIAG_IMPROVED_NAME)

sddae_epoch_seconds = sddae_probe_seconds / 2
ss_epoch_seconds = ss_probe_seconds
full_estimated_seconds = (
    (default_pretrain_epochs + default_finetune_epochs) * sddae_epoch_seconds
    + default_ss_epochs * ss_epoch_seconds
)
target_seconds = TARGET_TOTAL_MINUTES * 60
scale = min(1.0, target_seconds / full_estimated_seconds)

PRETRAIN_EPOCHS = max(1, round(default_pretrain_epochs * scale))
FINETUNE_EPOCHS = max(1, round(default_finetune_epochs * scale))
SS_EPOCHS = max(1, round(default_ss_epochs * scale))

estimated_selected_seconds = (
    (PRETRAIN_EPOCHS + FINETUNE_EPOCHS) * sddae_epoch_seconds
    + SS_EPOCHS * ss_epoch_seconds
)

BASELINE_NAME = f'sddae_r_L{WINDOW_SIZE}_d{QUALITY_DELAY}_z{LATENT_DIM}_r{LABEL_RATIO:g}'
IMPROVED_NAME = f'ss_ddfae_L{WINDOW_SIZE}_d{QUALITY_DELAY}_z{LATENT_DIM}_r{LABEL_RATIO:g}'

diagnosis = {
    **runtime,
    'target_total_minutes': TARGET_TOTAL_MINUTES,
    'sddae_probe_seconds': round(sddae_probe_seconds, 3),
    'ss_probe_seconds': round(ss_probe_seconds, 3),
    'sddae_epoch_seconds': round(sddae_epoch_seconds, 3),
    'ss_epoch_seconds': round(ss_epoch_seconds, 3),
    'full_estimated_minutes': round(full_estimated_seconds / 60, 2),
    'recommended_estimated_minutes': round(estimated_selected_seconds / 60, 2),
    'scale': round(scale, 3),
    'label_ratio': LABEL_RATIO,
    'window_size': WINDOW_SIZE,
    'quality_delay': QUALITY_DELAY,
    'latent_dim': LATENT_DIM,
    'pretrain_epochs': PRETRAIN_EPOCHS,
    'finetune_epochs': FINETUNE_EPOCHS,
    'ss_epochs': SS_EPOCHS,
}

display(pd.DataFrame([diagnosis]))
print('S-DDAE 输出名称:', BASELINE_NAME)
print('SS-DDFAE 输出名称:', IMPROVED_NAME)

## 7. 训练 S-DDAE 基线模型

S-DDAE 先通过重构任务学习动态窗口表示，再使用有标签样本微调质量变量预测头。训练轮数和标签比例由上一节诊断结果推荐。

In [ ]:
!python -m deep_quality.cli.train_sddae \
  --config configs/sddae_single_scale.yaml \
  --label-ratio {LABEL_RATIO} \
  --window-size {WINDOW_SIZE} \
  --quality-delay {QUALITY_DELAY} \
  --latent-dim {LATENT_DIM} \
  --pretrain-epochs {PRETRAIN_EPOCHS} \
  --finetune-epochs {FINETUNE_EPOCHS} \
  --output-name {BASELINE_NAME}

## 8. 训练 SS-DDFAE 半监督模型

SS-DDFAE 读取单尺度 S-DDAE checkpoint，迁移兼容参数，然后同时使用有标签样本和无标签样本训练。核心机制包括多层特征预测、注意力融合、一致性约束和伪标签。

In [ ]:
!python -m deep_quality.cli.train_ss_ddfae \
  --config configs/ss_ddfae.yaml \
  --baseline-checkpoint artifacts/checkpoints/{BASELINE_NAME}.pt \
  --label-ratio {LABEL_RATIO} \
  --window-size {WINDOW_SIZE} \
  --quality-delay {QUALITY_DELAY} \
  --latent-dim {LATENT_DIM} \
  --epochs {SS_EPOCHS} \
  --output-name {IMPROVED_NAME}

## 9. 对两个模型做后处理搜索

后处理会在验证集上比较 `raw`、`ema`、`ar`、`ema+ar`，选择验证集 RMSE 最低的方法，并保存到 `artifacts/tables/*_postprocess.json`。

In [ ]:
!python -m deep_quality.cli.tune_postprocess \
  --checkpoint artifacts/checkpoints/{BASELINE_NAME}.pt

!python -m deep_quality.cli.tune_postprocess \
  --checkpoint artifacts/checkpoints/{IMPROVED_NAME}.pt

## 10. 后处理评估

评估命令会加载 checkpoint，重新构造数据处理流程，读取后处理摘要，并输出后处理前后的测试集指标。

In [ ]:
!python -m deep_quality.cli.evaluate_checkpoint \
  --checkpoint artifacts/checkpoints/{BASELINE_NAME}.pt

!python -m deep_quality.cli.evaluate_checkpoint \
  --checkpoint artifacts/checkpoints/{IMPROVED_NAME}.pt

## 11. 在线推理模拟

在线推理模拟会逐条遍历测试窗口，记录单样本推理耗时，并输出平均延迟。

In [ ]:
!python -m deep_quality.cli.simulate_online_inference \
  --checkpoint artifacts/checkpoints/{BASELINE_NAME}.pt

!python -m deep_quality.cli.simulate_online_inference \
  --checkpoint artifacts/checkpoints/{IMPROVED_NAME}.pt

## 12. 汇总训练、后处理和在线推理指标

重点看：

1. `raw_rmse` 到 `rmse` 的变化，表示后处理带来的误差改善。
2. `SS-DDFAE` 与 `S-DDAE` 的后处理后 RMSE、MAE、R2 对比。
3. `avg_latency_ms`，表示在线场景单样本推理耗时。
4. `rmse_vs_sddae`、`mae_vs_sddae`、`r2_vs_sddae`，表示相对 S-DDAE 基线的变化。

In [ ]:
import json
import pandas as pd
from pathlib import Path

def load_json(path):
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)

rows = []
for name in [BASELINE_NAME, IMPROVED_NAME]:
    train_metrics = load_json(f'artifacts/tables/{name}_metrics.json')
    eval_metrics = load_json(f'artifacts/tables/{name}_eval_metrics.json')
    online_metrics = load_json(f'artifacts/tables/{name}_online_metrics.json')
    rows.append({
        'model': train_metrics['model'],
        'label_ratio': train_metrics['label_ratio'],
        'raw_rmse': eval_metrics['raw_rmse'],
        'post_rmse': eval_metrics['rmse'],
        'post_mae': eval_metrics['mae'],
        'post_r2': eval_metrics['r2'],
        'postprocess_method': eval_metrics['postprocess_method'],
        'avg_latency_ms': online_metrics['avg_latency_ms']
    })

summary_df = pd.DataFrame(rows)
baseline_rmse = float(summary_df.loc[summary_df['model'] == 'sddae_r', 'post_rmse'].iloc[0])
baseline_mae = float(summary_df.loc[summary_df['model'] == 'sddae_r', 'post_mae'].iloc[0])
baseline_r2 = float(summary_df.loc[summary_df['model'] == 'sddae_r', 'post_r2'].iloc[0])
summary_df['rmse_vs_sddae'] = summary_df['post_rmse'] - baseline_rmse
summary_df['mae_vs_sddae'] = summary_df['post_mae'] - baseline_mae
summary_df['r2_vs_sddae'] = summary_df['post_r2'] - baseline_r2
display(summary_df)

best_row = summary_df.sort_values(['post_rmse', 'post_mae'], ascending=[True, True]).iloc[0]
print('后处理后 RMSE 最低模型:', best_row['model'])
print('后处理后 RMSE:', best_row['post_rmse'])

## 13. 展示预测曲线、散点图和残差图

预测曲线用于展示模型能否跟踪质量变量变化；散点图用于展示预测值和真实值的一致性；残差图用于观察误差是否存在明显偏移或趋势。

In [ ]:
from IPython.display import Image, display

for name in [BASELINE_NAME, IMPROVED_NAME]:
    print('\n' + '=' * 80)
    print(name)
    print('=' * 80)
    for suffix in ['prediction', 'scatter', 'residuals', 'eval_prediction', 'eval_scatter', 'eval_residuals', 'online_prediction']:
        path = Path(f'artifacts/figures/{name}_{suffix}.png')
        if path.exists():
            print(path)
            display(Image(filename=str(path)))

## 14. 查看输出文件清单

所有训练产物都在 `artifacts/` 下。展示时优先打开 `tables` 中的指标文件和 `figures` 中的图像文件。

In [ ]:
!find artifacts -maxdepth 2 -type f | sort

## 15. 可选：多尺度 S-DDAE 展示

多尺度 S-DDAE 同时使用不同窗口长度和采样步长建模动态信息。SS-DDFAE 当前只支持单尺度基线 checkpoint，因此多尺度部分作为 S-DDAE 扩展实验单独展示。

In [ ]:
MULTISCALE_NAME = 'sddae_r_ms40x1-24x2-12x4_d12_z52_r1'

!python -m deep_quality.cli.train_sddae \
  --config configs/sddae.yaml \
  --label-ratio 1.0 \
  --quality-delay 12 \
  --latent-dim 52 \
  --scales 40x1,24x2,12x4 \
  --output-name {MULTISCALE_NAME}

## 16. 打包并保存到 Google Drive

运行后会将 `artifacts/` 打包为 zip，并保存到 Google Drive 的 `MyDrive/DeepQuality/` 目录中。该压缩包包含 checkpoint、指标表、预测 CSV 和图片。

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

drive_output_dir = Path('/content/drive/MyDrive/DeepQuality')
drive_output_dir.mkdir(parents=True, exist_ok=True)

archive_path = shutil.make_archive('/content/deepquality_artifacts', 'zip', 'artifacts')
drive_archive_path = drive_output_dir / 'deepquality_artifacts.zip'
shutil.copy2(archive_path, drive_archive_path)

print('已保存到 Google Drive:', drive_archive_path)